# 08 · Model size: does a bigger RoFormer help?
Setup fixed to the chosen configuration: **200 ms window, 50 ms step, patch-16 embedding (25 tokens), absolute + delta heads,
Kalman fusion**. Only the model's size changes, one factor at a time, plus two combinations:

| name | change from baseline | params | dropout | weights (10-bit) | fits Zynq-7020 BRAM | current HLS |
|---|---|---|---|---|---|---|
| `base` | none (reuses notebook 07's `w200ms_patch16_t25` if attached) | 160 k | 0.10 | 1.6 Mbit | yes | yes |
| `base_seed1` | same, different random seed: **noise reference** | 160 k | 0.10 | 1.6 Mbit | yes | yes |
| `L2` | 2 encoder layers | 292 k | 0.15 | 2.9 Mbit | yes | needs a layer loop |
| `L3` | 3 encoder layers | 425 k | 0.20 | 4.3 Mbit | no | no |
| `F512` | FFN 256 → 512 | 226 k | 0.15 | 2.3 Mbit | yes | yes |
| `D192` | width 128 → 192, FFN 384 (head dim 24) | 338 k | 0.15 | 3.4 Mbit | yes, barely | yes |
| `D256` | width 256, FFN 512 (head dim 32) | 582 k | 0.20 | 5.8 Mbit | no | yes |
| `L2_F512` | 2 layers and FFN 512 | 424 k | 0.20 | 4.2 Mbit | no | needs a layer loop |

Dropout grows with size (0.10 up to 200 k parameters, 0.15 up to 400 k, 0.20 above) so a larger model is not penalised for
over-fitting it could have avoided. A larger model therefore differs from `base` in both size and regularisation; that is the
intended comparison ("best version of each size"). Set the environment variable `EMG_SIZE_DROPOUT=0` to train all at 0.10.

Everything else is identical to notebook 07 (float training, normalised loss, selection on the absolute head, filter re-tuned per model).
A change counts as a real gain only if it beats `base` by more than `base_seed1` differs from `base`, in most subjects.

**Before running:** re-upload `emg_common.py` (adds `hw_cost`, faster bootstrap). Attach NinaPro DB2 **and notebook 07's output**
(reuses its preprocessed data and baseline checkpoint). **Turn the GPU on** and check the first cell prints `cuda True`.
Expected time 4 to 5 hours. Resumable: attach this notebook's own output and re-run. Click **Save Version** at the end.

In [ ]:
import os, sys, glob, json, time
for p in ['/kaggle/usr/lib/notebooks/quddusikashaf/emg_common', '/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '.', '..', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
assert hasattr(ec, 'hw_cost') and hasattr(ec, 'crop_last'), 're-upload the updated emg_common.py utility script'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')
N_BOOT = int(os.environ.get('EMG_NBOOT', 2000)); EPOCHS = int(os.environ.get('EMG_EPOCHS', 40))
OUT = f'{WORK}/scale'; os.makedirs(f'{OUT}/outputs', exist_ok=True)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| emg_common', ec.__file__)
if not torch.cuda.is_available() and 'EMG_WORK' not in os.environ:
    print('WARNING: no GPU. Settings > Accelerator > GPU T4, then restart; on CPU this notebook takes over a day.')

## 1 · Data: 200 ms windows, 50 ms step (reused from notebook 07 when attached)

In [ ]:
WIN, STEP = 400, 100
TAG = f'db2_processed_w{WIN}_s{STEP}'
BASE = os.environ.get('EMG_BASE_PATH')
if BASE is None:
    hits = sorted(glob.glob('/kaggle/input/**/DB2_s1/DB2_s1', recursive=True))
    BASE = os.path.dirname(os.path.dirname(hits[0])) if hits else ec.Config.base_path
dcfg = ec.Config(base_path=BASE, processed_dir=f'{WORK}/{TAG}', seq_len=WIN, stride=STEP,
                 n_subjects=int(os.environ.get('EMG_N_SUBJECTS', 10)), causal_filter=True, train_reps=(1, 3, 4, 6), test_reps=(2, 5))
reuse = sorted(glob.glob(f'/kaggle/input/**/{TAG}', recursive=True))
subjects = ec.list_subjects(dcfg) if not reuse else sorted(os.path.basename(os.path.dirname(f)) for f in glob.glob(f'{reuse[0]}/*/data.npz'))
npz_files = []
for s in subjects[:dcfg.n_subjects]:
    found = [f'{d}/{s}/data.npz' for d in [dcfg.processed_dir] + reuse if os.path.exists(f'{d}/{s}/data.npz')]
    npz_files.append(found[0] if found else ec.process_subject(dcfg, s))
print(len(npz_files), 'subjects from', os.path.dirname(os.path.dirname(npz_files[0])))

## 2 · Train (resumable)

In [ ]:
# Dropout grows with model size so larger models are not simply more over-fitted:
#   <= 200 k params: 0.10 (same as notebook 07)   200-400 k: 0.15   > 400 k: 0.20
# Set EMG_SIZE_DROPOUT=0 to train every model at 0.10 instead.
EXPERIMENTS = [
    dict(name='base',       reuse='w200ms_patch16_t25', dropout=0.10),   # 160 k
    dict(name='base_seed1', seed=1,                      dropout=0.10),   # 160 k
    dict(name='L2',         num_layers=2,                dropout=0.15),   # 292 k
    dict(name='L3',         num_layers=3,                dropout=0.20),   # 425 k
    dict(name='F512',       d_ff=512,                    dropout=0.15),   # 226 k
    dict(name='D192',       d_model=192, d_ff=384,       dropout=0.15),   # 338 k
    dict(name='D256',       d_model=256, d_ff=512,       dropout=0.20),   # 582 k
    dict(name='L2_F512',    num_layers=2, d_ff=512,      dropout=0.20),   # 424 k
]
if os.environ.get('EMG_SIZE_DROPOUT', '1') == '0':
    for e in EXPERIMENTS: e['dropout'] = 0.10
if 'EMG_EXPERIMENTS' in os.environ:
    keep = os.environ['EMG_EXPERIMENTS'].split(','); EXPERIMENTS = [e for e in EXPERIMENTS if e['name'] in keep]

def make_cfg(e):
    c = ec.Config(processed_dir=os.path.dirname(os.path.dirname(npz_files[0])), save_dir=f'{OUT}/outputs',
                  variant='patch', patch_stride=16, seq_len=WIN, stride=STEP,
                  heads='both', cond_prev=False, unroll_steps=0, d_model=128, d_ff=256, num_heads=8, num_layers=1,
                  norm='layernorm', dropout=0.1, epochs=EPOCHS, patience=8, lr=3e-4, batch_size=64, qat_epochs=0,
                  act_fmt=(10, 3), resid_fmt=(16, 8), weight_fmt=(10, 3), loss_norm=True, select_metric='val_abs',
                  seed=e.get('seed', 42), use_wandb=os.environ.get('EMG_WANDB', '1') == '1', project_name='ninapro_db2_scaling')
    for k in ('num_layers', 'd_model', 'd_ff', 'num_heads', 'dropout'):
        if k in e: setattr(c, k, e[k])
    c.apply_variant(); c.run_name = f"scale_{e['name']}"
    return c

device = ec.get_device(); models = {}; train_minutes = {}
for e in EXPERIMENTS:
    c = make_cfg(e)
    cost = ec.hw_cost(c)
    print(f"\n=== {e['name']}: layers {c.num_layers}, width {c.d_model}, FFN {c.d_ff}, dropout {c.dropout:.2f} | params {cost['params']:,} | "
          f"MACs {cost['MACs_M']:.2f} M | weights {cost['weights_mbit']:.2f} Mbit | fits on chip: {cost['fits_on_chip']}")
    names = [c.run_name] + ([e['reuse']] if 'reuse' in e else [])
    found = [p for n in names for p in [f'{OUT}/outputs/{n}.pt'] + sorted(glob.glob(f'/kaggle/input/**/{n}.pt', recursive=True)) if os.path.exists(p)]
    if found:
        m, c_loaded, extra = ec.load_checkpoint(found[0], device); print('reusing', found[0], extra.get('phase'))
        assert (c_loaded.num_layers, c_loaded.d_model, c_loaded.d_ff, c_loaded.seq_len, c_loaded.patch_stride, c_loaded.heads, c_loaded.cond_prev, round(c_loaded.dropout, 3)) == \
               (c.num_layers, c.d_model, c.d_ff, c.seq_len, c.patch_stride, c.heads, c.cond_prev, round(c.dropout, 3)), 'reused checkpoint has a different configuration'
        c = c_loaded
    else:
        run = ec.init_wandb(c)
        t0 = time.time(); m, info = ec.train_model(c, npz_files, run=run, tag=c.run_name)
        train_minutes[e['name']] = (time.time() - t0) / 60; print(f"trained in {train_minutes[e['name']]:.1f} min")
        if run is not None: run.finish()
    models[e['name']] = (m.eval(), c)

## 3 · Evaluate (re-tuned filter, per-subject bootstrap) and compare

In [ ]:
rows, per_subject = [], []
done_csv = f'{OUT}/summary.csv'
for name, (m, c) in models.items():
    t0 = time.time()
    preds = [ec.predict_run(m, r, c, device, feedback=True) for f in npz_files for r in ec.ordered_runs(f, 'test') if len(r['angles']) >= 4]
    gk = ec.kalman_sweep(preds, use_delta=True, init='abs'); gs = ec.kalman_sweep(preds, use_delta=False, init='abs')
    parse = lambda key: (float(key[0].split('=')[1]), float(key[1].split('=')[1]))
    kqr, sqr = parse(gk.stack().idxmin()), parse(gs.stack().idxmin())
    df = ec.per_subject_eval(m, npz_files, c, device, init='abs', kalman_qr=kqr, smooth_qr=sqr, n_boot=N_BOOT, tag=name)
    per_subject.append(df)
    a = df[df['subject'] == 'ALL'].iloc[0]; cost = ec.hw_cost(c, m)
    rows.append({'model': name, 'layers': c.num_layers, 'width': c.d_model, 'ffn': c.d_ff, 'dropout': c.dropout, **cost,
                 'train_min': train_minutes.get(name), 'abs_only_r2': a['abs_only_r2'], 'smoothed_r2': a['abs_smoothed_r2'],
                 'fused_r2': a['kalman_r2'], 'fused_r2_lo': a['kalman_r2_lo'], 'fused_r2_hi': a['kalman_r2_hi'],
                 'fused_rmse': a['kalman_rmse'], 'delta_gain': a['delta_gain'], 'kalman_qr': kqr})
    print(f"{name}: fused R2 {a['kalman_r2']:.4f} [{a['kalman_r2_lo']:.4f}, {a['kalman_r2_hi']:.4f}] RMSE {a['kalman_rmse']:.4f} "
          f"| abs-only {a['abs_only_r2']:.4f} | eval {time.time() - t0:.0f}s")
    pd.DataFrame(rows).to_csv(done_csv, index=False); pd.concat(per_subject, ignore_index=True).to_csv(f'{OUT}/per_subject.csv', index=False)
summary = pd.DataFrame(rows); per_subject = pd.concat(per_subject, ignore_index=True)
display(summary.round(4))

In [ ]:
# ---- paired, per-subject comparison against `base`; `base_seed1` sets the noise level ----
rm = per_subject[per_subject['subject'] != 'ALL'].pivot(index='subject', columns='setting', values='kalman_rmse')
r2 = per_subject[per_subject['subject'] != 'ALL'].pivot(index='subject', columns='setting', values='kalman_r2')
rng = np.random.RandomState(0); comp = []
for col in rm.columns:
    if col == 'base': continue
    d = (rm[col] - rm['base']).values
    boot = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(5000)]
    comp.append({'model': col, 'mean_rmse_change': d.mean(), 'ci_lo': np.percentile(boot, 2.5), 'ci_hi': np.percentile(boot, 97.5),
                 'subjects_better': int((d < 0).sum()), 'mean_r2_change': (r2[col] - r2['base']).mean()})
comp = pd.DataFrame(comp).merge(summary[['model', 'params', 'MACs_M', 'weights_mbit', 'latency_ms_U8', 'fits_on_chip', 'hls_supported']], on='model')
noise = abs(comp.loc[comp['model'] == 'base_seed1', 'mean_rmse_change'].values[0]) if (comp['model'] == 'base_seed1').any() else np.nan
comp['beats_seed_noise'] = (comp['mean_rmse_change'] < -noise) & (comp['ci_hi'] < 0)
comp.to_csv(f'{OUT}/paired_vs_base.csv', index=False)
print(f'seed noise (|base_seed1 - base| mean RMSE change): {noise:.4f}')
display(comp.round(4))
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for fits, mk in ((True, 'o'), (False, 'x')):
    s = summary[summary['fits_on_chip'] == fits]
    ax[0].errorbar(s['params'] / 1e3, s['fused_r2'], yerr=[s['fused_r2'] - s['fused_r2_lo'], s['fused_r2_hi'] - s['fused_r2']], fmt=mk, capsize=3,
                   label='fits Zynq-7020 BRAM' if fits else 'does not fit')
for _, r in summary.iterrows(): ax[0].annotate(r['model'], (r['params'] / 1e3, r['fused_r2']), fontsize=8, xytext=(4, 4), textcoords='offset points')
ax[0].set_xscale('log'); ax[0].set_xlabel('parameters (thousands)'); ax[0].set_ylabel('fused R² (pooled, 95 % CI)'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].barh(comp['model'], comp['mean_rmse_change'], xerr=[comp['mean_rmse_change'] - comp['ci_lo'], comp['ci_hi'] - comp['mean_rmse_change']], capsize=3)
ax[1].axvline(0, color='k', lw=1)
if np.isfinite(noise): ax[1].axvspan(-noise, noise, color='gray', alpha=.2, label='seed noise')
ax[1].set_xlabel('mean per-subject RMSE change vs base (negative = better)'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT}/scaling.png', dpi=120); plt.show()
json.dump({'seed_noise': noise, 'summary': summary.to_dict(orient='records'), 'paired': comp.to_dict(orient='records')},
          open(f'{OUT}/scaling_summary.json', 'w'), indent=2, default=str)

### Reading the result
* Look at `paired_vs_base.csv`: a size change is worth it only if `beats_seed_noise` is True **and** most subjects improve.
* Then check cost: `fits_on_chip` (weights under 3.5 Mbit of the 4.9 Mbit block RAM) and `latency_ms_U8` against the 50 ms step.
  `hls_supported` is False for more than one layer: the v2 HLS would need a loop over layers with per-layer weights.
* If only a non-fitting model helps, the gain is an upper bound on what extra capacity buys, not a deployable option as-is.